# Rally financiero 1 — análisis de series de tiempo
[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/clases-y-otras-tonteras/blob/rally-uno/rally_1/analisis_series_tiempo_colab.ipynb)

Corte fijo: **24 de septiembre de 2026**. Se analiza el rendimiento diario del USD/MXN y su relación con S&P 500, VIX y Treasury a 10 años. El notebook separa claramente el pronóstico ex ante (ARIMA) del análisis condicional con factores observados (SARIMAX).

In [ ]:
!pip -q install yfinance statsmodels


In [ ]:
import os
import random
import warnings

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels
import statsmodels.api as sm

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("default")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Seed:", SEED)
print("Python compatible con Colab")
print("yfinance:", yf.__version__)
print("statsmodels:", statsmodels.__version__)


## 1. Descarga y preparación

Se usan únicamente fechas con observaciones disponibles para las cuatro series. No se rellenan cierres faltantes con `ffill`, para evitar crear variaciones cero artificiales. El corte se mantiene fijo para que una nueva ejecución del notebook no cambie la muestra.

In [ ]:
START = "2020-01-01"
END = "2026-09-25"   # yfinance interpreta end como límite superior exclusivo

tickers = ["MXN=X", "^GSPC", "^VIX", "^TNX"]
rename = {
    "MXN=X": "USDMXN",
    "^GSPC": "SP500",
    "^VIX": "VIX",
    "^TNX": "TNX10Y"
}

raw = yf.download(
    tickers,
    start=START,
    end=END,
    auto_adjust=False,
    progress=False,
    threads=False
)

if raw.empty:
    raise RuntimeError("Yahoo Finance no devolvió datos.")

if isinstance(raw.columns, pd.MultiIndex):
    if "Close" not in raw.columns.get_level_values(0):
        raise RuntimeError("No se encontró el campo Close en la descarga.")
    prices = raw["Close"].copy()
else:
    # Caso defensivo si yfinance cambia el formato para una descarga
    prices = raw.copy()

prices = prices.rename(columns=rename)

required = ["USDMXN", "SP500", "VIX", "TNX10Y"]
missing = [c for c in required if c not in prices.columns]
if missing:
    raise RuntimeError(f"Faltan columnas en la descarga: {missing}")

prices = (
    prices[required]
    .apply(pd.to_numeric, errors="coerce")
    .dropna(how="any")
    .sort_index()
)

prices = prices.loc[~prices.index.duplicated(keep="first")]

if len(prices) < 500:
    raise RuntimeError("La muestra descargada es demasiado corta para el ejercicio.")

assert prices.index.is_monotonic_increasing
assert prices.index.is_unique
assert prices.index.max() <= pd.Timestamp("2026-09-24")

data = pd.DataFrame(index=prices.index)
data["usdmxn_ret"] = 100 * np.log(prices["USDMXN"]).diff()
data["sp500_ret"] = 100 * np.log(prices["SP500"]).diff()
data["vix_change"] = prices["VIX"].diff()
data["tnx_change"] = prices["TNX10Y"].diff()

data = data.replace([np.inf, -np.inf], np.nan).dropna()

print("Periodo:", prices.index.min().date(), "a", prices.index.max().date())
print("Observaciones de precios:", len(prices))
print("Observaciones del modelo:", len(data))
display(data.describe().T)
display(data.corr())


## 2. Inspección gráfica y estacionariedad

USD/MXN y S&P 500 se expresan como rendimientos logarítmicos porcentuales. VIX y Treasury se trabajan en primeras diferencias. En niveles, la prueba ADF incluye constante y tendencia; en rendimientos incluye constante.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), prices.columns):
    ax.plot(prices.index, prices[col], linewidth=1)
    ax.set_title(col)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

def adf_result(series, regression):
    r = adfuller(series.dropna(), regression=regression, autolag="AIC")
    return {
        "ADF": r[0],
        "p_value": r[1],
        "rezagos": r[2],
        "n_obs": r[3]
    }

adf_table = pd.DataFrame({
    "USD/MXN nivel": adf_result(prices["USDMXN"], "ct"),
    "USD/MXN rendimiento": adf_result(data["usdmxn_ret"], "c"),
    "S&P 500 nivel": adf_result(prices["SP500"], "ct"),
    "S&P 500 rendimiento": adf_result(data["sp500_ret"], "c")
}).T

display(adf_table)


## 3. División temporal y preparación para `statsmodels`

El último 20% se reserva para evaluación fuera de muestra. Las fechas se conservan por separado para las tablas y gráficas, pero **los modelos reciben un `RangeIndex`**. Esto evita el error de `statsmodels` con índices de fechas irregulares sin frecuencia fija.

In [ ]:
split = int(len(data) * 0.80)

train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

assert train.index.max() < test.index.min()

train_dates = train.index.copy()
test_dates = test.index.copy()

# Índices enteros para statsmodels
y_train = train["usdmxn_ret"].reset_index(drop=True).astype(float)
y_test = test["usdmxn_ret"].reset_index(drop=True).astype(float)

xcols = ["sp500_ret", "vix_change", "tnx_change"]
X_train = train[xcols].reset_index(drop=True).astype(float)
X_test = test[xcols].reset_index(drop=True).astype(float)

assert isinstance(y_train.index, pd.RangeIndex)
assert isinstance(X_train.index, pd.RangeIndex)
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5)
    }

print("Train:", len(y_train), "| Test:", len(y_test))
print("Primera fecha test:", test_dates.min().date())


## 4. ARIMA univariado

Se prueban órdenes `(p,0,q)` con `p,q = 0,...,3`, incluyendo `(0,0,0)`. La selección se hace por AIC **solamente dentro de la muestra de entrenamiento**. Se descartan estimaciones que no convergen o que producen un AIC no finito.

In [ ]:
candidates = []

for p in range(4):
    for q in range(4):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                fit = ARIMA(
                    y_train,
                    order=(p, 0, q),
                    trend="c"
                ).fit()

            converged = bool(fit.mle_retvals.get("converged", True))
            aic = float(fit.aic)

            if converged and np.isfinite(aic):
                candidates.append((p, q, aic))
        except Exception:
            pass

aic_table = (
    pd.DataFrame(candidates, columns=["p", "q", "AIC"])
    .sort_values("AIC")
    .reset_index(drop=True)
)

if aic_table.empty:
    raise RuntimeError("Ningún ARIMA convergió correctamente.")

best_p = int(aic_table.loc[0, "p"])
best_q = int(aic_table.loc[0, "q"])
best_order = (best_p, 0, best_q)

arima = ARIMA(y_train, order=best_order, trend="c").fit()

arima_raw = np.asarray(
    arima.forecast(steps=len(y_test)),
    dtype=float
)

arima_pred = pd.Series(arima_raw, index=test_dates, name="ARIMA")

zero_pred = pd.Series(0.0, index=test_dates, name="Benchmark cero")

print("ARIMA seleccionado:", best_order)
display(aic_table.head(10))

lb = acorr_ljungbox(arima.resid, lags=[10, 20], return_df=True)
print("Ljung-Box de residuos ARIMA:")
display(lb)


## 5. SARIMAX condicional

El SARIMAX incorpora S&P 500, VIX y Treasury observados en cada fecha. Por ello esta evaluación mide **capacidad explicativa/condicional** y no debe presentarse como un pronóstico ex ante puro. Para evitar problemas numéricos, se intenta el orden ARIMA seleccionado y, si no converge, se recorre la tabla de órdenes válidos hasta encontrar una especificación SARIMAX convergente.

In [ ]:
sarimax = None
sarimax_order = None

for row in aic_table.itertuples(index=False):
    order_try = (int(row.p), 0, int(row.q))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            fit_try = SARIMAX(
                y_train,
                exog=X_train,
                order=order_try,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=1000)

        converged = bool(fit_try.mle_retvals.get("converged", True))
        params_ok = np.isfinite(np.asarray(fit_try.params, dtype=float)).all()

        if converged and params_ok:
            sarimax = fit_try
            sarimax_order = order_try
            break
    except Exception:
        continue

if sarimax is None:
    raise RuntimeError("No se encontró una especificación SARIMAX convergente.")

sarimax_raw = np.asarray(
    sarimax.get_forecast(
        steps=len(y_test),
        exog=X_test
    ).predicted_mean,
    dtype=float
)

sarimax_pred = pd.Series(
    sarimax_raw,
    index=test_dates,
    name="SARIMAX condicional"
)

comparison = pd.DataFrame([
    {"modelo": "Benchmark cero", **metrics(y_test, zero_pred)},
    {"modelo": f"ARIMA{best_order}", **metrics(y_test, arima_pred)},
    {"modelo": f"SARIMAX{sarimax_order} condicional", **metrics(y_test, sarimax_pred)}
]).set_index("modelo")

print("Orden SARIMAX:", sarimax_order)
display(comparison)

window = min(150, len(test))
plt.figure(figsize=(13, 5))
plt.plot(test_dates[-window:], test["usdmxn_ret"].iloc[-window:], label="Observado", linewidth=1.2)
plt.plot(arima_pred.index[-window:], arima_pred.iloc[-window:], label="ARIMA", linewidth=1)
plt.plot(sarimax_pred.index[-window:], sarimax_pred.iloc[-window:], label="SARIMAX condicional", linewidth=1)
plt.axhline(0, linewidth=0.8)
plt.title("Validación fuera de muestra — rendimiento diario USD/MXN")
plt.ylabel("% diario")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 6. Sensibilidad contemporánea del USD/MXN

Se estima OLS con errores estándar HAC/Newey-West. El coeficiente del S&P 500 se interpreta como **asociación parcial contemporánea** manteniendo constantes VIX, Treasury y el rezago del USD/MXN. No es causalidad ni una beta CAPM.

In [ ]:
reg = data.copy()
reg["lag_usdmxn"] = reg["usdmxn_ret"].shift(1)
reg = reg.dropna()

X = sm.add_constant(
    reg[["sp500_ret", "vix_change", "tnx_change", "lag_usdmxn"]]
)
y = reg["usdmxn_ret"]

ols_hac = sm.OLS(y, X).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 5}
)

print(ols_hac.summary())

beta_sp = float(ols_hac.params["sp500_ret"])
p_sp = float(ols_hac.pvalues["sp500_ret"])
ci_sp = ols_hac.conf_int().loc["sp500_ret"]

sensitivity = pd.DataFrame({
    "coeficiente": [beta_sp],
    "p_value": [p_sp],
    "IC95_inf": [float(ci_sp.iloc[0])],
    "IC95_sup": [float(ci_sp.iloc[1])]
}, index=["S&P 500"])

display(sensitivity)

if p_sp < 0.05 and beta_sp < 0:
    print("Asociación parcial negativa y significativa al 5%.")
elif p_sp < 0.05:
    print("Asociación parcial positiva y significativa al 5%.")
else:
    print("El coeficiente parcial del S&P 500 no es significativo al 5%.")

rolling_beta = (
    data["usdmxn_ret"].rolling(60).cov(data["sp500_ret"])
    / data["sp500_ret"].rolling(60).var()
)

plt.figure(figsize=(13, 4))
plt.plot(rolling_beta.index, rolling_beta, linewidth=1)
plt.axhline(0, linewidth=0.8)
plt.title("Sensibilidad móvil USD/MXN vs. S&P 500 — 60 sesiones")
plt.ylabel("Coeficiente móvil")
plt.grid(alpha=0.25)
plt.show()


## 7. Pronóstico univariado y validaciones finales

Para el pronóstico final se usa sólo ARIMA, porque un pronóstico ex ante no conoce todavía los valores futuros del S&P 500, VIX y Treasury. El modelo se vuelve a estimar con toda la muestra disponible y se proyectan cinco sesiones.

In [ ]:
y_full = data["usdmxn_ret"].reset_index(drop=True).astype(float)

final_arima = ARIMA(
    y_full,
    order=best_order,
    trend="c"
).fit()

forecast_obj = final_arima.get_forecast(steps=5)
forecast_frame = forecast_obj.summary_frame(alpha=0.05)[
    ["mean", "mean_ci_lower", "mean_ci_upper"]
].copy()

forecast_frame.index = [f"sesion_{i}" for i in range(1, 6)]
forecast_frame.columns = [
    "pronostico_pct",
    "limite_inferior_95",
    "limite_superior_95"
]

display(forecast_frame)

checks = pd.Series({
    "seed_fijada": SEED == 42,
    "fechas_ordenadas": prices.index.is_monotonic_increasing,
    "fechas_unicas": prices.index.is_unique,
    "sin_NA_precios": not prices.isna().any().any(),
    "sin_NA_modelo": not data.isna().any().any(),
    "corte_correcto": prices.index.max() <= pd.Timestamp("2026-09-24"),
    "train_antes_test": train.index.max() < test.index.min(),
    "rangeindex_endog": isinstance(y_train.index, pd.RangeIndex),
    "rangeindex_exog": isinstance(X_train.index, pd.RangeIndex),
    "indices_modelo_alineados": y_train.index.equals(X_train.index),
    "arima_parametros_finitos": np.isfinite(np.asarray(arima.params, dtype=float)).all(),
    "sarimax_parametros_finitos": np.isfinite(np.asarray(sarimax.params, dtype=float)).all(),
    "predicciones_arima_finitas": np.isfinite(arima_pred.to_numpy()).all(),
    "predicciones_sarimax_finitas": np.isfinite(sarimax_pred.to_numpy()).all(),
    "metricas_finitas": np.isfinite(comparison[["MAE", "RMSE"]].to_numpy()).all(),
    "beta_finita": np.isfinite(beta_sp),
    "forecast_final_finito": np.isfinite(forecast_frame.to_numpy()).all()
}, name="OK")

display(checks)

if not checks.all():
    failed = checks.index[~checks].tolist()
    raise AssertionError(f"Fallaron validaciones: {failed}")

print("Validaciones superadas.")
